In [ ]:
!pip install "pydantic-ai[mcp]"
!pip install "pydantic-ai[fastmcp]"

In [ ]:
!sudo apt-get install zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [4]:
!nohup ollama serve > ollama.log 2>&1 &

In [5]:
!ollama run gpt-oss “What is the capital of the Turkey?”


Thinking...
User asks: "What is the capital of the Turkey?" They may mean "the country 
of Turkey" (capital: Ankara). Alternatively could refer to "Turkey" as a bi
bird. The question phrasing ambiguous. Probably they want the capital city 
of Turkey, i.e., Ankara. Provide answer and maybe mention Istanbul historic
historically but not official. Also clarify that the word Turkey is also an
an animal; its 'capital' not applicable. So respond: The capital of Turkey 
(the country) is Ankara.

Should be brief.
...done thinking.

The capital of **Turkey** (the country in Western Asia/Europe) is **Ankara*
**Ankara**.  

*(If you were referring to the bird “turkey,” it doesn’t have a capital—tho
capital—those are cities for countries, not animals.)*



In [6]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) # We get some warnings related to notebook

In [7]:
import asyncio
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai import Agent, DeferredToolRequests, DeferredToolResults

ollama_model = OpenAIChatModel(
model_name='gpt-oss',
provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)

agent = Agent(
    model=ollama_model,
)

In [9]:
result = await agent.run('Write a short poem on computer science')

In [10]:
print(result.output)

In silicon caves the silent thoughts ignite—  
Logic gates pulse like a heartbeat’s byte,  
Patterns woven in binary moonlight.  

Algorithms dance through endless night,  
Debugging whispers, “Find the hidden clue”;  
Computers breathe: the future coded true.


In [11]:
from fastmcp import FastMCP
from pydantic_ai.toolsets.fastmcp import FastMCPToolset


fastmcp_server = FastMCP('my_server')

#Lets make up a code calculation method so that we know our model can't know its own knowledge to guess (we will be sure that it uses the mcp tool)
@fastmcp_server.tool()
async def bixby_code(a: int) -> str:
    return str(10 * a + 55) + "A" * (a%2) + "B" * ((a+1) %2)

toolset = FastMCPToolset(fastmcp_server)
approval_required_toolset = toolset.approval_required(lambda ctx, tool_def, tool_args: tool_def)


agent = Agent(
    model=ollama_model,
    toolsets=[approval_required_toolset],
    output_type=[str, DeferredToolRequests],
)

In [39]:
result = await agent.run('What is the bixby code of 10?')
print(result.output)

DeferredToolRequests(calls=[], approvals=[ToolCallPart(tool_name='bixby_code', args='{"a":10}', tool_call_id='call_45c7uhl8')], metadata={})


In [40]:
messages = result.all_messages()

In [42]:
result = await agent.run(
  message_history=messages,
  deferred_tool_results=DeferredToolResults(
      approvals={
          [x.tool_call_id for x in result.output.approvals if x.tool_name == "bixby_code"][0]: True,
      }
  )
)

In [43]:
print(result.output)

155B


In [45]:
result = await agent.run('What is the bixby code of 10?')
print(result.output)
messages = result.all_messages()

DeferredToolRequests(calls=[], approvals=[ToolCallPart(tool_name='bixby_code', args='{"a":10}', tool_call_id='call_8vsnspku')], metadata={})


In [46]:
result = await agent.run(
  message_history=messages,
  deferred_tool_results=DeferredToolResults(
      approvals={
          [x.tool_call_id for x in result.output.approvals if x.tool_name == "bixby_code"][0]: False,
      }
  )
)

In [47]:
print(result.output)

I’m sorry, but I can’t help with that.


In [50]:
messages

[ModelRequest(parts=[UserPromptPart(content='What is the bixby code of 10?', timestamp=datetime.datetime(2026, 6, 20, 11, 47, 27, 182288, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 6, 20, 11, 47, 27, 182717, tzinfo=datetime.timezone.utc), run_id='019ee4db-7149-7723-bade-a613c5b80c1b', conversation_id='019ee4db-7149-7723-bade-a61284347c0e'),
 ModelResponse(parts=[ThinkingPart(content='We need to answer this question. The user: "What is the bixby code of 10?" We have defined a function in the namespace "functions.bixby_code" that takes JSON input {a:number} and returns any.\n\nThe likely desired action: call the function with a=10. Then output result? As ChatGPT, we must comply with tool usage guidelines. The question might want to compute something: Bixby code of 10 maybe means applying some transformation rule known to system. But we have no other instructions. The developer message defines a tool functions.bixby_code that probably implements the transformation 